# 1 · The 5-D data model

Everything in `meteoraster` is one class, `MeteoRaster`, wrapping one 5-D array plus the
metadata needed to read, subset, aggregate, save and plot it. This notebook is about that
array: what each axis means, how the coordinates are stored, and how to tell whether the
cube is actually full.

**Needs:** core dependencies only (numpy, pandas, xarray, h5netcdf).

In [1]:
import importlib.util
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Environment noise, nothing to do with meteoraster: xarray probes every NetCDF/GRIB
# backend it can find (cfgrib needs the ecCodes binary), and this conda env's h5py and
# HDF5 builds differ by a patch release.
warnings.filterwarnings('ignore', message='(?s).*ecCodes.*')
warnings.filterwarnings('ignore', message='(?s).*h5py is running against HDF5.*')

# Report warnings without their absolute path, so the committed output does not carry
# one machine's directory layout. The warnings themselves stay visible.
def brief_warning(message, category, filename, lineno, line=None):
    return '%s:%d: %s: %s\n' % (Path(filename).name, lineno, category.__name__, message)


warnings.formatwarning = brief_warning

# Run straight from a clone: if `meteoraster` is not installed in this environment,
# fall back to the repository root one level up.
if importlib.util.find_spec('meteoraster') is None:
    sys.path.insert(0, str(Path('..').resolve()))

from meteoraster import MeteoRaster

RESOURCES = Path('..') / 'tests' / 'resources'

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 60)

## The five axes

| Axis | Name | Meaning |
|---|---|---|
| 0 | `production_datetime` | when the forecast was **issued** |
| 1 | `ensemble_member` | ensemble member index, 0-based |
| 2 | `leadtime` | forecast horizon, a `pd.Timedelta` (or a `pd.DateOffset` for monthly data) |
| 3 | `latitude` (`y`) | rows, always north-first |
| 4 | `longitude` (`x`) | columns |

The vocabulary that matters for the rest of these notebooks:

> **production date** is when the forecast was made, **leadtime** is how far ahead it
> looks, and **event date = production date + leadtime** is the moment being forecast.

One forecast issue is a row; reading across that row walks forward in time.

In [2]:
tiny = MeteoRaster(
    data=np.arange(2 * 3 * 2 * 4 * 5, dtype=float).reshape(2, 3, 2, 4, 5),
    latitudes=np.linspace(40, 30, 4),                       # 1-D is fine, see below
    longitudes=np.linspace(-10, 10, 5),
    production_datetime=pd.to_datetime(['2023-01-01', '2023-01-02']).values,
    leadtimes=np.array([pd.Timedelta(days=1), pd.Timedelta(days=2)]),
    units='mm',
    variable='precip',
    verbose=False,
)

tiny.data.shape

(2, 3, 2, 4, 5)

A small helper used throughout these notebooks. Note the `pd.to_timedelta` call: the
leadtimes are stored as numpy `timedelta64[ns]`, which prints as
`10800000000000 nanoseconds` rather than `0 days 03:00:00`. Converting before display is
the difference between a readable table and an unreadable one.

In [3]:
def axes_table(mr):
    """One row per axis: its name, its size, and the span of its labels."""
    prod = pd.to_datetime(mr.production_datetime)
    if isinstance(mr.leadtimes[0], pd.DateOffset):
        lead = list(mr.leadtimes)                 # monthly/seasonal: DateOffsets, not durations
    else:
        lead = pd.to_timedelta(mr.leadtimes)

    rows = [
        ('production_datetime', mr.data.shape[0], '%s  ...  %s' % (prod[0], prod[-1])),
        ('ensemble_member', mr.data.shape[1], '0  ...  %d' % (mr.data.shape[1] - 1)),
        ('leadtime', mr.data.shape[2], '%s  ...  %s' % (lead[0], lead[-1])),
        ('latitude (y)', mr.data.shape[3],
         '%.2f  ...  %.2f N' % (mr.latitudes[0, 0], mr.latitudes[-1, 0])),
        ('longitude (x)', mr.data.shape[4],
         '%.2f  ...  %.2f E' % (mr.longitudes[0, 0], mr.longitudes[0, -1])),
    ]
    table = pd.DataFrame(rows, columns=['axis name', 'size', 'from  ...  to'])
    table.index.name = 'axis'
    return table


axes_table(tiny)

,axis name,size,from ... to
axis,,,
0,production_datetime,2,2023-01-01 00:00:00 ... 2023-01-02 00:00:00
1,ensemble_member,3,0 ... 2
2,leadtime,2,1 days 00:00:00 ... 2 days 00:00:00
3,latitude (y),4,40.00 ... 30.00 N
4,longitude (x),5,-10.00 ... 10.00 E


## Coordinates are stored as 2-D grids

`latitudes` and `longitudes` are always 2-D arrays the same shape as one spatial slice,
even when you pass 1-D vectors — the constructor meshes them. That is what lets the class
carry curvilinear grids without a special case.

In [4]:
print('given   latitudes: 1-D, %d values' % 4)
print('stored  latitudes:', tiny.latitudes.shape, ' longitudes:', tiny.longitudes.shape)
print()
print('latitudes (each row constant):')
print(tiny.latitudes)
print()
print('longitudes (each column constant):')
print(tiny.longitudes)

given   latitudes: 1-D, 4 values
stored  latitudes: (4, 5)  longitudes: (4, 5)

latitudes (each row constant):
[[40.         40.         40.         40.         40.        ]
 [36.66666667 36.66666667 36.66666667 36.66666667 36.66666667]
 [33.33333333 33.33333333 33.33333333 33.33333333 33.33333333]
 [30.         30.         30.         30.         30.        ]]

longitudes (each column constant):
[[-10.  -5.   0.   5.  10.]
 [-10.  -5.   0.   5.  10.]
 [-10.  -5.   0.   5.  10.]
 [-10.  -5.   0.   5.  10.]]


## Two things the constructor fixes for you

**Longitudes are wrapped** from a `0 … 360` convention to `-180 … 180`, and the data is
rolled to follow. **Latitudes are flipped** to descending order if they arrive
south-first, so `data[..., 0, :]` is always the northernmost row.

In [5]:
raw_lons = np.array([0.0, 90.0, 180.0, 270.0])
greenwich = MeteoRaster(
    data=np.array([0.0, 1.0, 2.0, 3.0]).reshape(1, 1, 1, 1, 4),
    latitudes=np.array([0.0]),
    longitudes=raw_lons,
    production_datetime=pd.to_datetime(['2023-01-01']).values,
    leadtimes=np.array([pd.Timedelta(0)]),
    verbose=False,
)

pd.DataFrame({
    'longitude in': raw_lons,
    'value in': [0.0, 1.0, 2.0, 3.0],
    'longitude out': greenwich.longitudes.ravel(),
    'value out': greenwich.data.ravel(),
})

,longitude in,value in,longitude out,value out
0,0.0,0.0,-90.0,3.0
1,90.0,1.0,0.0,0.0
2,180.0,2.0,90.0,1.0
3,270.0,3.0,180.0,2.0


In [6]:
south_first = MeteoRaster(
    data=np.arange(3, dtype=float).reshape(1, 1, 1, 3, 1),
    latitudes=np.array([30.0, 40.0, 50.0]),                 # ascending: south first
    longitudes=np.array([0.0]),
    production_datetime=pd.to_datetime(['2023-01-01']).values,
    leadtimes=np.array([pd.Timedelta(0)]),
    verbose=False,
)

print('latitudes in :', [30.0, 40.0, 50.0], ' values in :', [0.0, 1.0, 2.0])
print('latitudes out:', south_first.latitudes.ravel().tolist(),
      ' values out:', south_first.data.ravel().tolist())
print('\nnorth-first, and the data followed the flip.')

latitudes in : [30.0, 40.0, 50.0]  values in : [0.0, 1.0, 2.0]
latitudes out: [50.0, 40.0, 30.0]  values out: [2.0, 1.0, 0.0]

north-first, and the data followed the flip.


## A real forecast

`tethys_NOAA_GFS_0.25_2026.02.02.nct` is a converted Tethys file: seven daily NOAA GFS
issues over Belgium, 3-hourly out to 16 days, on a 0.25° grid. It is deterministic, so the
ensemble axis has size 1.

In [7]:
gfs = MeteoRaster.load(RESOURCES / 'tethys_NOAA_GFS_0.25_2026.02.02.nct', verbose=False)

print('variable: %s   units: %s' % (gfs.variable, gfs.units))
axes_table(gfs)

variable: PRATE   units: mm/3h


,axis name,size,from ... to
axis,,,
0,production_datetime,7,2026-02-02 00:00:00 ... 2026-02-08 00:00:00
1,ensemble_member,1,0 ... 0
2,leadtime,129,0 days 00:00:00 ... 16 days 00:00:00
3,latitude (y),29,54.00 ... 47.00 N
4,longitude (x),57,-2.00 ... 12.00 E


The leadtime axis is uniform here — 129 columns, three hours apart, reaching 16 days:

In [8]:
leads = pd.to_timedelta(gfs.leadtimes)
print('first five :', [str(lead) for lead in leads[:5]])
print('last       :', leads[-1], ' = %d days' % (leads[-1].total_seconds() / 86400))
print('spacing    :', {str(step): int(n)
                       for step, n in pd.Series(np.diff(leads)).value_counts().items()})

first five : ['0 days 00:00:00', '0 days 03:00:00', '0 days 06:00:00', '0 days 09:00:00', '0 days 12:00:00']
last       : 16 days 00:00:00  = 16 days
spacing    : {'0 days 03:00:00': 128}


## An ensemble, and calendar leadtimes

`tethys_c3s_ecmwf51_tprate_2026.07.nct` is an ECMWF SEAS5 seasonal forecast: **one**
production, **eight** members, and six leadtimes that are *calendar months* rather than
fixed durations. Those are `pd.DateOffset` objects — "one month" is not a fixed number of
hours, and the library keeps that distinction rather than flattening it to 30 days.

In [9]:
seas5 = MeteoRaster.load(RESOURCES / 'tethys_c3s_ecmwf51_tprate_2026.07.nct', verbose=False)

print('variable: %s   units: %s' % (seas5.variable, seas5.units))
print('leadtime type:', type(seas5.leadtimes[0]).__name__, '->', list(seas5.leadtimes[:3]))
axes_table(seas5)

variable: tprate   units: mm/month


leadtime type: DateOffset -> [<DateOffset: months=1>, <DateOffset: months=2>, <DateOffset: months=3>]


,axis name,size,from ... to
axis,,,
0,production_datetime,1,2026-07-01 00:00:00 ... 2026-07-01 00:00:00
1,ensemble_member,8,0 ... 7
2,leadtime,6,<DateOffset: months=1> ... <DateOffset: months=6>
3,latitude (y),33,52.50 ... 20.50 N
4,longitude (x),45,-19.50 ... 24.50 E


So the same 5-D container covers very different datasets, by letting axes collapse to
size 1:

| Dataset | production | member | leadtime | reads as |
|---|---|---|---|---|
| GFS above | 7 | **1** | 129 | a deterministic forecast archive |
| SEAS5 above | **1** | 8 | 6 | one seasonal ensemble |
| reanalysis / observations | many | **1** | **1** | a plain time series of maps |

In [10]:
pd.DataFrame(
    [(gfs.data.shape[0], gfs.data.shape[1], gfs.data.shape[2]),
     (seas5.data.shape[0], seas5.data.shape[1], seas5.data.shape[2])],
    index=['NOAA GFS', 'ECMWF SEAS5'],
    columns=['productions', 'members', 'leadtimes'],
)

,productions,members,leadtimes
NOAA GFS,7,1,129
ECMWF SEAS5,1,8,6


## Event time

The value at row *p*, column *l* describes the interval **`[p + l, p + l + step)`** — the
stamp *opens* the interval, a forward convention. So a single production date maps onto a
whole stretch of event dates, and different productions overlap on the same event date.
That overlap is exactly what makes a forecast archive useful, and what notebook 5 plots.

In [11]:
first_three = pd.to_datetime(gfs.production_datetime)[:3]
some_leads = pd.to_timedelta(gfs.leadtimes)[[0, 1, 8, 16]]

pd.DataFrame(
    [[prod + lead for lead in some_leads] for prod in first_three],
    index=pd.Index(first_three, name='production date'),
    columns=pd.Index(some_leads, name='leadtime'),
)

leadtime,0 days 00:00:00,0 days 03:00:00,1 days 00:00:00,2 days 00:00:00
production date,,,,
2026-02-02,2026-02-02,2026-02-02 03:00:00,2026-02-03,2026-02-04
2026-02-03,2026-02-03,2026-02-03 03:00:00,2026-02-04,2026-02-05
2026-02-04,2026-02-04,2026-02-04 03:00:00,2026-02-05,2026-02-06


## Is the cube actually full?

Real archives have holes: a missed download, a run that failed. Three methods report on
that, and this file has a real hole in it — one of the seven productions is entirely
missing.

In [12]:
print('is_complete():', gfs.is_complete())
print('missing fraction of the whole cube: %.4f  (= 1/7)' % np.isnan(gfs.data).mean())

is_complete(): False
missing fraction of the whole cube: 0.1429  (= 1/7)


`get_complete_index()` is a production × leadtime boolean table. Summing it per production
shows which issue is empty:

In [13]:
complete = gfs.get_complete_index()
summary = pd.DataFrame({
    'complete leadtimes': complete.sum(axis=1),
    'of': complete.shape[1],
})
summary['status'] = np.where(summary['complete leadtimes'] == 0, '<-- entirely missing', 'ok')
summary

,complete leadtimes,of,status
production_datetime,,,
2026-02-02,129,129,ok
2026-02-03,129,129,ok
2026-02-04,129,129,ok
2026-02-05,129,129,ok
2026-02-06,129,129,ok
2026-02-07,0,129,<-- entirely missing
2026-02-08,129,129,ok


`get_missing()` is finer grained — the fraction of NaN pixels for every
(leadtime, ensemble_member) against every production. Here is a corner of it:

In [14]:
missing = gfs.get_missing()
print('shape:', missing.shape, ' rows:', missing.index.names, ' columns:', missing.columns.name)
missing.iloc[:4, :].round(3)

shape: (129, 7)  rows: ['leadtime', 'ensemble_member']  columns: production_datetime


,production_datetime,2026-02-02,2026-02-03,2026-02-04,2026-02-05,2026-02-06,2026-02-07,2026-02-08
leadtime,ensemble_member,,,,,,,
0 days 00:00:00,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
0 days 03:00:00,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
0 days 06:00:00,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
0 days 09:00:00,0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


---
**Next:** [02 · Loading, joining, saving and xarray](02_load_join_save_xarray.ipynb) —
getting cubes in and out of files.